<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Evaluating_LLMs_Exercises_W7D4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [ ]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List
import pandas as pd

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []
    for batch in batch_generator(texts, batch_size):
        inputs = tokenizer(["summarize: " + t for t in batch], return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(inputs.input_ids, max_new_tokens=max_new_tokens)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        summaries.extend(decoded)

        torch.cuda.empty_cache()
        gc.collect()

    return summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = True
if RUN_T5:
    print(f"Generating summaries for {len(train_df)} rows...")
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=4)
    results_df = pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    })
    display(results_df.head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")


### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    # Join sentences with newlines as recommended for ROUGE-L
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    preds_norm = [normalize_text(p) for p in preds]
    refs_norm = [normalize_text(r) for r in refs]
    return rouge.compute(predictions=preds_norm, references=refs_norm, use_stemmer=True)

# Smoke test
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check:")
print(compute_rouge_score(test_preds, test_refs))


### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    summaries = []
    for batch in batch_generator(texts, batch_size):
        # Simple TL;DR prompt
        prompts = [t[:500] + "\n\nTL;DR:" for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

        # Only decode the newly generated tokens
        for i in range(len(batch)):
            input_len = inputs.input_ids[i].shape[0]
            gen_text = tokenizer.decode(outputs[i][input_len:], skip_special_tokens=True)
            summaries.append(gen_text.strip())

        torch.cuda.empty_cache()
        gc.collect()
    return summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    results = []
    for p, r in zip(df[pred_col], df[ref_col]):
        score = rouge.compute(predictions=[normalize_text(p)], references=[normalize_text(r)], use_stemmer=True)
        results.append(score['rougeL'])
    return results

RUN_COMPARE = True
if RUN_COMPARE and 'train_summaries_t5' in locals():
    print("Comparing models...")
    # Generate base model summaries
    train_summaries_t5_base = summarize_with_t5(train_df['prompt_text'].tolist()[:10], model_name='t5-base', batch_size=2)
    train_summaries_gpt2 = summarize_with_gpt2(train_df['prompt_text'].tolist()[:10], batch_size=2)

    compare_df = train_df.head(10).copy()
    compare_df['t5_small'] = train_summaries_t5[:10]
    compare_df['t5_base'] = train_summaries_t5_base
    compare_df['gpt2'] = train_summaries_gpt2

    compare_df['rougeL_t5_small'] = compute_rouge_per_row(compare_df, 't5_small')
    compare_df['rougeL_t5_base'] = compute_rouge_per_row(compare_df, 't5_base')
    compare_df['rougeL_gpt2'] = compute_rouge_per_row(compare_df, 'gpt2')

    display(compare_df[['t5_small', 't5_base', 'gpt2', 'rougeL_t5_small', 'rougeL_t5_base', 'rougeL_gpt2']].head())


### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
def compare_models(results_df):
    metrics = ['rougeL_t5_small', 'rougeL_t5_base', 'rougeL_gpt2']
    avg_scores = results_df[metrics].mean()
    return pd.DataFrame(avg_scores, columns=['Mean ROUGE-L'])

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    return df[['prompt_title'] + pred_cols]

if 'compare_df' in locals():
    print("Aggregate Comparison Table:")
    display(compare_models(compare_df))
    print("\nSide-by-Side Summaries:")
    display(compare_models_summaries(compare_df, ['t5_small', 't5_base', 'gpt2']).head(3))


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
